# Chapter 3 — Bayesian Updating: Recursive Belief Refinement

**Companion notebook for _Bayesian Methods for Autonomous Robotics and Intelligent Infrastructure Systems_**

**Credits.** Book author and scientific/editorial direction: **Khalid M. Mosalam** (UC Berkeley). Original companion notebook development: **Fan Hu**; additional companion notebooks and shared computational implementation: **Ziad K. Mosalam**. This Chapter 3 notebook extends the existing shared example with additional sequential, biased-model, and correlated-observation illustrations. See the repository [README](../README.md) and Git commit history for project-wide contributions and subsequent revisions.

This is a **synthetic** computational illustration, not a field deployment or measured infrastructure dataset.

**Event definitions.** $S$ denotes an actual stop sign; $\neg S$ denotes no stop sign. $D_i$ means observation $i$ reports a stop sign, while $\neg D_i$ means it reports no stop sign. A negative detector report **does not** establish that the sign is absent.

The baseline prior is $P(S)=0.6$, sensitivity $P(D_i\mid S)=0.9$, and false-positive rate $P(D_i\mid\neg S)=0.2$. Unless otherwise stated, successive reports are assumed **conditionally independent given the true state**. These illustrative probabilities are not measured camera characteristics.

Run all cells from the repository root or \`notebooks/\` directory. Outputs are saved under \`figures/\` and \`results/\`. The original shared example generator is retained for compatibility.

In [ ]:
from pathlib import Path
import json
import math
import sys
import matplotlib.pyplot as plt

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(root))
plt.style.use(root / "notebooks" / "book.mplstyle")
(root / "figures").mkdir(exist_ok=True)
(root / "results").mkdir(exist_ok=True)

from robotics_book.chapter_examples import run_example

# Retain the existing chapter example and its files.
baseline = run_example("bayesian-updating", root=root)
print("Original five-positive sequence:", [round(p, 4) for p in baseline["values"]["positive_updates"]])

## 1. Updating with positive and negative detector reports

For prior $p=P(S)$, a positive report uses the conditional likelihoods $(0.9,0.2)$ under $(S,\neg S)$, whereas a negative report uses $(0.1,0.8)$. The returned posterior is always the **probability that a stop sign is actually present**, rather than the probability that the detector reports a sign.

In [ ]:
PRIOR = 0.60
SENSITIVITY = 0.90
FALSE_POSITIVE_RATE = 0.20

def bayes_update(prior, positive, sensitivity=SENSITIVITY,
                 false_positive_rate=FALSE_POSITIVE_RATE):
    """Return P(stop sign present | observations so far)."""
    if not (0 <= prior <= 1 and 0 <= sensitivity <= 1
            and 0 <= false_positive_rate <= 1):
        raise ValueError("Probabilities must lie between 0 and 1.")
    like_sign = sensitivity if positive else 1 - sensitivity
    like_no_sign = (false_positive_rate if positive
                    else 1 - false_positive_rate)
    evidence = like_sign * prior + like_no_sign * (1 - prior)
    if evidence == 0:
        raise ValueError("Observed report has zero probability under the model.")
    return like_sign * prior / evidence

first_positive = bayes_update(PRIOR, True)
second_positive = bayes_update(first_positive, True)
second_negative = bayes_update(first_positive, False)

print(f"Prior:                     {PRIOR:.4f}")
print(f"After D1 (positive):       {first_positive:.4f}")
print(f"After D1, D2 (+,+):        {second_positive:.4f}")
print(f"After D1, not D2 (+,-):    {second_negative:.4f}")

assert math.isclose(first_positive, 0.54 / 0.62, abs_tol=1e-12)
assert math.isclose(second_positive, 0.81 * 0.6 / (0.81 * 0.6 + 0.04 * 0.4), abs_tol=1e-12)
assert math.isclose(second_negative, 0.09 * 0.6 / (0.09 * 0.6 + 0.16 * 0.4), abs_tol=1e-12)

## 2. Alternative second observations

The next plot **branches** after the first positive observation. The second positive and second negative reports are **alternative scenarios**, not successive reports. The posterior falls after a negative report even though the detector can miss an actual sign.

In [ ]:
fig, ax = plt.subplots(figsize=(5.8, 3.5), layout="constrained")
ax.plot([0, 1, 2], [PRIOR, first_positive, second_positive],
        color="#003262", marker="o", linewidth=2, label="Second report positive")
ax.plot([1, 2], [first_positive, second_negative],
        color="#B17A00", marker="s", linestyle="--", linewidth=2,
        label="Second report negative")
ax.set_xticks([0, 1, 2], ["Prior", "First +", "Second report"])
ax.set_ylim(0, 1.08)
ax.set_ylabel("Probability of stop sign")
ax.legend(loc="lower left", fontsize=8)
ax.grid(axis="y")
for x, y, color, shift in [(0, PRIOR, "#003262", .035),
                            (1, first_positive, "#003262", .035),
                            (2, second_positive, "#003262", -.07),
                            (2, second_negative, "#B17A00", .035)]:
    ax.text(x, y + shift, f"{y:.3f}", ha="center", color=color, fontsize=9)
fig.savefig(root / "figures" / "bayesian-updating-sequences.pdf",
            metadata={"CreationDate": None, "ModDate": None})
fig.savefig(root / "figures" / "bayesian-updating-sequences.png", dpi=200)
plt.show()

## 3. Five positive reports: correctly specified versus biased sensor model

Under conditional independence, posterior odds are multiplied by the likelihood ratio $0.9/0.2=4.5$ for every positive report. If an incorrectly optimistic model assumes a false-positive rate of $0.05$, it uses a likelihood ratio of $18$. These are different **inference models applied to the same hypothetical sequence**, not evidence that the biased model performs better.

In [ ]:
def positive_history(count, false_positive_rate):
    values = [PRIOR]
    for _ in range(count):
        values.append(bayes_update(values[-1], True,
                                   false_positive_rate=false_positive_rate))
    return values

correct_history = positive_history(5, 0.20)
biased_history = positive_history(5, 0.05)
assert math.isclose(correct_history[1], first_positive, abs_tol=1e-12)
assert math.isclose(correct_history[2], second_positive, abs_tol=1e-12)
assert math.isclose(biased_history[2], 0.997946611909651, abs_tol=1e-12)

fig, ax = plt.subplots(figsize=(5.8, 3.5), layout="constrained")
ax.plot(range(6), correct_history, color="#003262", marker="o",
        label="Specified false-positive rate: 0.20")
ax.plot(range(6), biased_history, color="#B17A00", marker="s",
        linestyle="--", label="Biased assumed rate: 0.05")
ax.set(xlabel="Number of positive reports", ylabel="Posterior probability",
       ylim=(0.55, 1.03))
ax.set_xticks(range(6))
ax.grid(axis="y")
ax.legend(loc="lower right", fontsize=8)
fig.savefig(root / "figures" / "bayesian-updating-bias.pdf",
            metadata={"CreationDate": None, "ModDate": None})
fig.savefig(root / "figures" / "bayesian-updating-bias.png", dpi=200)
plt.show()
print("Correct model, 2 positives:", round(correct_history[2], 4))
print("Biased model, 2 positives: ", round(biased_history[2], 4))

## 4. Correlated damage-detector observations (Exercise 3.3)

This is a **separate synthetic structural-damage example**, not part of the stop-sign sequence. Let $H$ denote damage, with prior $P(H)=0.30$. Although the marginal positive-detection probabilities are $P(D_i\mid H)=0.80$ and $P(D_i\mid\neg H)=0.20$, the **joint** probabilities of two positive reports are $P(D_1,D_2\mid H)=0.70$ and $P(D_1,D_2\mid\neg H)=0.12$. Using products of the marginal likelihoods would incorrectly assume conditional independence.

### Numerical comparison (two positive observations)

| Updating assumption | Likelihood under damage | Likelihood under no damage | Posterior probability of damage |
|:--|--:|--:|--:|
| Specified joint likelihoods (correct) | 0.70 | 0.12 | **0.7143** |
| Product of marginal likelihoods (incorrect independence) | 0.64 | 0.04 | **0.8727** |

The incorrect independence assumption gives a posterior approximately **0.1584 higher** in this example. The table summarizes the reproducible calculation below; the analytical solution appears in **Appendix E, Exercise 3.3**, of the book. The bar plot is retained as an optional notebook illustration, not proposed for the printed chapter.


In [ ]:
damage_prior = 0.30
joint_damage, joint_no_damage = 0.70, 0.12
correct_correlated = (joint_damage * damage_prior /
                      (joint_damage * damage_prior +
                       joint_no_damage * (1 - damage_prior)))
incorrect_independent = ((0.80 ** 2) * damage_prior /
                         ((0.80 ** 2) * damage_prior +
                          (0.20 ** 2) * (1 - damage_prior)))
assert math.isclose(correct_correlated, 0.7142857142857143, abs_tol=1e-12)
assert math.isclose(incorrect_independent, 0.8727272727272728, abs_tol=1e-12)

fig, ax = plt.subplots(figsize=(5.8, 3.5), layout="constrained")
bars = ax.bar([0, 1], [correct_correlated, incorrect_independent],
              color=["#003262", "#B17A00"], width=0.55)
ax.bar_label(bars, fmt="%.3f", padding=3)
ax.set_xticks([0, 1], ["Specified joint\nlikelihoods",
                        "Incorrect independence\nassumption"])
ax.set(ylabel="Posterior probability of damage", ylim=(0, 1.02))
ax.grid(axis="y")
fig.savefig(root / "figures" / "bayesian-updating-correlation.pdf",
            metadata={"CreationDate": None, "ModDate": None})
fig.savefig(root / "figures" / "bayesian-updating-correlation.png", dpi=200)
plt.show()
print(f"Correlated: {correct_correlated:.4f}; incorrectly independent: {incorrect_independent:.4f}")

## 5. Reader explorations and limitations

1. Change the sequence to **positive, negative, positive**, and compare the posterior with three positives. Explain each change in probability.
2. Change the initial prior from 0.60 to 0.05. Which values change, and which detector characteristics stay fixed?
3. Vary the **assumed** false-positive rate; compare the posterior after two positive reports with the nominal calculation. Explain why additional observations cannot, by themselves, repair a persistently biased likelihood model.
4. Compare the correlated and incorrectly independent results in Exercise 3.3. Why is their difference not evidence of real-world detector performance?

**Scope.** This notebook uses declared synthetic probabilities only. It does not establish real detector calibration, actual conditional independence of repeated camera frames, or operational safety. Duplicating an observation is not new evidence. Correlation should be represented using joint/conditional likelihoods or explicit shared latent variables.

In [ ]:
summary = {
    "stop_sign": {
        "prior": PRIOR,
        "first_positive": first_positive,
        "second_positive": second_positive,
        "second_negative": second_negative,
        "five_positive_correct": correct_history,
        "five_positive_biased": biased_history,
    },
    "correlated_damage": {
        "specified_joint_likelihoods": correct_correlated,
        "incorrect_independence": incorrect_independent,
    },
    "figures": [
        "bayesian-updating-sequences.pdf",
        "bayesian-updating-bias.pdf",
        "bayesian-updating-correlation.pdf",
    ],
}
(root / "results" / "bayesian-updating-extended.json").write_text(
    json.dumps(summary, indent=2) + "\n"
)
print("Saved the extended results and three figures.")